In [1]:
from lila import mock_lya

lya_mock = mock_lya.MockLya(
    noiseless_file="/projects/pbreysse/lya_forest_CO/lya_storage/Lyman-alpha-ASTRID_z2.5_unzipped/Lyman-alpha-ASTRID_z2.5/map_ASTRID_true_0.25_z2.5.hdf5",
    spec_file="/projects/pbreysse/lya_forest_CO/lya_storage/Lyman-alpha-ASTRID_z2.5_unzipped/Lyman-alpha-ASTRID_z2.5/spectra_ASTRID_noiseless_z2.5_1000_voxels.hdf5",
    boxsize=[250, 250, 250],   # cMpc/h
    dperp=2.5,                 # mean transverse sightline separation in cMpc/h
                               # (e.g. LATIS=2.5, PFS=3.7, eBOSS=13, DESI=10)
    sn=2,                      # average S/N per Angstrom
    transpose=(1, 0, 2),       # axis reordering if needed to match CO box orientation
    # transpose = (2,0,1),
    silent_mode=False,
)


[ 000000.00 ]   0: 10-04 18:40  MockLya         INFO     Starting
[ 000000.00 ]   0: 10-04 18:40  MockLya         INFO     dperp = 2.5


In [2]:
import h5py
import json
import numpy as np
import matplotlib.pyplot as plt
from nbodykit.lab import ArrayMesh, FFTPower

with h5py.File("/projects/pbreysse/lya_forest_CO/lya_storage/Lyman-alpha-ASTRID_z2.5_unzipped/Lyman-alpha-ASTRID_z2.5/map_ASTRID_true_0.25_z2.5.hdf5", "r") as f:
    d = f["map"]
    print("shape:", d.shape)
    print("dataset attrs:", dict(d.attrs))
    print("file attrs:", dict(f.attrs))
with np.load("/users/jiawenfan/work/lya_CO/limlam_cartesian_v2_simplified/co_li_cartesian.npz") as f:
    Tco = np.array(f["map_cube"], dtype=np.float64)


shape: (1000, 1000, 1000)
dataset attrs: {}
file attrs: {}


In [ ]:
    # Lyα auto-power using LiLa
    lya_stats = Stats(
        mock_lya=matched_lya,
        kmin=kmin,
        kmax=kmax,
        dk=dk,
    )
    auto_lya = lya_stats.get_lya_pk(mode="1d").power

    # CO auto-power and cross-power using identical FFT settings
    co_mesh = ArrayMesh(Tco - Tco.mean(), BoxSize=box)
    lya_mesh = ArrayMesh(dF, BoxSize=box)

    auto_co = FFTPower(
        first=co_mesh,
        **settings,
    ).power

    cross = FFTPower(
        first=co_mesh,
        second=lya_mesh,
        **settings,
    ).power

    # Calculate S/N with matching bins and Fourier mode counts
    result = signal_to_noise(
        k=np.asarray(cross["k"]),
        Pco=np.asarray(auto_co["power"].real),
        Plya=np.asarray(auto_lya["power"].real),
        Pcross=np.asarray(cross["power"].real),
        modes=np.asarray(cross["modes"]),
        noise_co=noise_co,
        noise_lya=noise_lya,
        volume_ratio=volume_ratio,
    )

    result["k_edges"] = np.asarray(cross.edges["k"])
    result["boxsize_cmpch"] = box
    result["volume_ratio"] = volume_ratio

    print(f"CO total S/N = {result['sn_CO_total']:.3f}")
    print(f"Lyα total S/N = {result['sn_F_total']:.3f}")
    print(f"Cross total S/N = {result['sn_cross_total']:.3f}")

    return result